# Transfer Learning

## Importing Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import matplotlib.pyplot as plt

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## Building file paths and labels

In [ ]:
LABELS_PATH = '../data/training_solutions_rev1.csv'
IMG_DIR = '../data/raw/images_training_rev1/images_training_rev1'
IMG_SIZE = 224
BATCH_SIZE = 32

labels_df = pd.read_csv(LABELS_PATH)
clean_df = labels_df[labels_df['Class1.3'] < 0.5].copy()

def assign_class(row):
    if row['Class1.1'] >= 0.5:
        return 'Smooth'
    elif row['Class4.1'] >= 0.5:
        return 'Spiral'
    else:
        return 'Irregular'

clean_df['label'] = clean_df.apply(assign_class, axis=1)

# build full file paths
clean_df['filepath'] = clean_df['GalaxyID'].apply(
    lambda gid: os.path.join(IMG_DIR, f"{gid}.jpg")
)

# encode labels
le = LabelEncoder()
clean_df['label_encoded'] = le.fit_transform(clean_df['label'])
print("Classes:", le.classes_)
print(clean_df['label'].value_counts())
print(f"Total: {len(clean_df)}")

## train/val/test split on file path

In [ ]:
filepaths = clean_df['filepath'].values
labels = clean_df['label_encoded'].values

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    filepaths, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {len(X_train_paths)}")
print(f"Val:   {len(X_val_paths)}")
print(f"Test:  {len(X_test_paths)}")

## Building tf.data Pipeline

In [ ]:
import tensorflow as tf

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    # no /255.0 — EfficientNet handles its own preprocessing internally
    return img, label

def build_dataset(filepaths, labels, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.map(
        load_and_preprocess,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = build_dataset(X_train_paths, y_train, shuffle=True)
val_ds = build_dataset(X_val_paths, y_val, shuffle=False)
test_ds = build_dataset(X_test_paths, y_test, shuffle=False)

print("Datasets built successfully.")

# quick sanity check - grab one batch and check shapes
for images, labels_batch in train_ds.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Label batch shape: {labels_batch.shape}")
    print(f"Pixel range: {images.numpy().min():.2f} - {images.numpy().max():.2f}")

With tf.data pipeline, we don't save processed arrays to Drive, i.e., images are loaded directly from raw files on demand during training. The file paths and label splits are lightweight and rebuilt each session.

In [ ]:
print("Using tf.data pipeline — no need to save processed arrays.")
print(f"Train batches: {len(train_ds)}")
print(f"Val batches:   {len(val_ds)}")
print(f"Test batches:  {len(test_ds)}")

## Building transfer learning model

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# load pretrained EfficientNetB0 without its top classification layer
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# freeze the base model first
base_model.trainable = False

# data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.5),
], name='augmentation')

from tensorflow.keras.applications.efficientnet import preprocess_input

#building the model
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Lambda(preprocess_input)(x)  # EfficientNet's own preprocessing
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

## Compiling and Training phase 1 (frozen backbone)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_phase1 = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Phase 1: Training classification head only (backbone frozen)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop_phase1]
)

## Unfreezing and fine-tuning (Phase 2)

In [ ]:
# unfreeze the base model
base_model.trainable = True

# recompile with much lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_phase2 = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Phase 2: Fine-tuning entire model (backbone unfrozen)...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop_phase2]
)

## Evaluating on test set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Plotting Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
train_loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

phase1_epochs = len(history_phase1.history['accuracy'])

axes[0].plot(train_acc, label='Train Accuracy')
axes[0].plot(val_acc, label='Val Accuracy')
axes[0].axvline(x=phase1_epochs, color='gray', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Transfer Learning Accuracy (EfficientNetB0)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(train_loss, label='Train Loss')
axes[1].plot(val_loss, label='Val Loss')
axes[1].axvline(x=phase1_epochs, color='gray', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Transfer Learning Loss (EfficientNetB0)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/transfer_learning_curves.png')
plt.show()

## Saving Trained Model

In [ ]:
model_save_path = '../models/transfer_learning_efficientnetb0.keras'
model.save(model_save_path)
print(f"Model saved: {model_save_path}")